In [15]:
from keras import layers
from keras import Input
from keras.models import Model

import numpy as np
import tqdm
import keras    
import tensorflow as tf
import os
import csv
import pathlib
import unicode

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator
# https://github.com/sieu-n/KoOCR-tensorflow/blob/main/utils/model_architectures.py

In [16]:
#PRE DEFINE

DATA_SIZE = 10000
VALID_DATA_SIZE = DATA_SIZE / 5

ORG_TRAIN_CSV_PATH = ""
SHUF_TRAIN_CSV_PATH = ""

ORG_VALID_CSV_PATH = ""
SHUF_TRAIN_CSV_PATH = ""

SAVE_DIR = "/root/Data/hangul/weights"
WEIGHT_FILE = SAVE_DIR + "/backbone1.weights.h5"
KERAS_FILE = SAVE_DIR + "/backbone1.keras"

In [17]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print(tf.__version__)
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

2.16.1


2024-05-28 16:15:38.348640: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-28 16:15:38.348725: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-28 16:15:38.348769: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-28 16:15:38.348937: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-28 16:15:38.348949: I tensorflow/core/common_runtime/gpu/gpu

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 5025571469140608789
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 2355888128
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 14592890734109654559
 physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [18]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18,
'0':19, '1':20, '2':21, '3':22, '4':23, '5':24, '6':25, '7':26, '8':27, '9':28, '-':29}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20, None:21}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ',
            19: '0', 20:'1', 21:'2', 22:'3', 23:'4', 24:'5', 25:'6', 26:'7', 27:'8', 28:'9', 29:'-'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ', 21:None}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

In [19]:

def getSpecificExtensionFiles(path, extension):
    out = []
    for (path, dir, files) in os.walk(path):
        for filename in files:
            ext = os.path.splitext(filename)[-1]
            if ext == extension:
                #print("%s/%s" % (path, filename))
                out.append(path + "/" + filename)
    return out

In [20]:
import matplotlib.pyplot as plt
import random

# draw_text = '람'
# font = "/root/Data/font/clova-all/가람연꽃/나눔손글씨_가람연꽃.ttf"

# fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")

def DataAugmentation():
    augment = tf.keras.Sequential([
        tf.keras.layers.experimental.preprocessing.RandomZoom( height_factor=(-0.2, 0.1),width_factor=(-0.2, 0.1),fill_mode='constant'),
        tf.keras.layers.experimental.preprocessing.RandomRotation(0.1,fill_mode='constant'),
        tf.keras.layers.experimental.preprocessing.RandomTranslation(0.1,0.1,fill_mode='constant')
        
    ])
    return augment

from PIL import Image,ImageDraw,ImageFont

def CreateFontImage(str, fontPath):
    font = ImageFont.truetype(fontPath, 28, encoding = 'utf-8')
    left, top, right, bottom = font.getbbox(str)
    width = right - left
    height = bottom - top
    
    canvas = Image.new('RGB', (width + 10, height + 14), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((3,3), str, 'black', font)
    
    #print(canvas)
    img = tf.image.convert_image_dtype(canvas, tf.float32)
    img = tf.image.resize(img, (64, 64))    
    img = np.array(img)
    img = np.expand_dims(img, axis=0)
    
    #plt.imshow(img)
    #print(img)

    # save the blank canvas to a file
    #canvas.save("unicode-text.png", "PNG")
    #canvas.show()
    return img

    
#CreateFontImage(draw_text, font)

def getFontImage(fontPath, imageNum):
    #fontFiles = getSpecificExtensionFiles(fontPath, ".ttf")
    
    #for i in range(1, imageNum):
        #fontidx = random.randrange(0, len(fontFiles) + 1)
    cho = random.randrange(0, 19)
    jung = random.randrange(0, 21)
    jong = random.randrange(0, 28)
    
    ja = label2ja[cho]
    mo = label2mo[jung]
    ba = label2ba[jong]

    char = unicode.join_jamos_char(ja, mo ,ba)
    #print(char)
    
    label1 = np.expand_dims(np.array(cho), axis=0)
    label2 = np.expand_dims(np.array(jung), axis=0)
    label3 = np.expand_dims(np.array(jong), axis=0)
    

    #yield CreateFontImage(char, fontFiles[fontidx])
    return CreateFontImage(char, fontPath), (label1, label2, label3)

In [21]:
synthImagePath = ""
basePath = "" + "/"

def getSynthDataset():
    cnt = 0
    
    os.os.system("shuf /root/Data/hangul/dataset/tranDataset.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    csvFile = open("")
    
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        imgFile = os.path.join(basePath, imgFile)
        
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        yield img, (label1, label2, label3)
        

        
        
        

In [22]:
def get_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    fontNum = len(fontFiles)
    
    os.system("shuf /root/Data/hangul/dataset/MergedData.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    csvFile = open("/root/Data/hangul/dataset/shuffled_tranDataset.csv", 'r', encoding='utf-8')
    #csvFile = open("/root/Data/hangul/dataset/test.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        yield img, (label1, label2, label3)
        
        fontidx = random.randrange(0, len(fontFiles))
        yield getFontImage(fontFiles[fontidx], 10)

        if (cnt > DATA_SIZE): break
        else                : cnt += 1

        
        
def get_valid_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    fontNum = len(fontFiles)

    os.system("shuf /root/Data/hangul/dataset/MergedValidData.csv > /root/Data/hangul/dataset/shuffled_validation.csv")    
    csvFile = open("/root/Data/hangul/dataset/shuffled_validation.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        yield img, (label1, label2, label3)
        
        fontidx = random.randrange(0, len(fontFiles))
        yield getFontImage(fontFiles[fontidx], 10)

        if (cnt > VALID_DATA_SIZE): break
        else                : cnt += 1

In [23]:
dataset = tf.data.Dataset.from_generator(get_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )


validDtaset = tf.data.Dataset.from_generator(get_valid_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )

In [24]:
REG_LAMBDA = 0 #0.01 # 0.001 0.1 0.05
REG_ON = 0
cl2_reg = tf.keras.regularizers.l2(REG_LAMBDA)

def AddSingleLayer(inputTensor, filters, kernel_size = (3,3)):
    if REG_ON:
        x = layers.Conv2D(filters, kernel_size, padding='same', kernel_regularizer=cl2_reg)(inputTensor)
    else:
        x = layers.Conv2D(filters, kernel_size, padding='same')(inputTensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    return x

def add_block(model, num_filters):
    #x = AddSingleLayer(model, num_filters)
    if REG_ON:
        x = layers.Conv2D(num_filters, (3, 3), padding='same', kernel_regularizer=cl2_reg)(model)
    else:
        x = layers.Conv2D(num_filters, (3, 3), padding='same')(model)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.DepthwiseConv2D(3,3, activation='relu')(x)
    # x = layers.BatchNormalization()
    
    return x
    
def BranchBlock(inputTensor, filters, layerSize, lastLayerName):
    # if REG_ON:
    #     x = layers.Conv2D(filters, (3, 3), padding='same', kernel_regularizer=cl2_reg)(inputTensor)
    # else:
    #     x = layers.Conv2D(filters, (3, 3), padding='same')(inputTensor)
    # x = layers.BatchNormalization()(x)
    # x = layers.Activation('relu')(x)
    #x = layers.MaxPooling2D(pool_size = 2)(x)
    #x = layers.SpatialDropout2D(0.1)(x)

    
    #x = layers.Flatten()(x)
    x = layers.Flatten()(inputTensor)
    
    if REG_ON:
        #x = layers.Dense(128, activation='softmax', kernel_regularizer=cl2_reg)(x)
        x = layers.Dense(layerSize, activation='softmax', name = lastLayerName, kernel_regularizer=cl2_reg)(x)
    else:
        #x = layers.Dense(128, activation='softmax')(x)
        x = layers.Dense(layerSize, activation='softmax', name = lastLayerName)(x)
    return x

    
def create_model():
    inputs = tf.keras.Input(shape=(64,64,3), dtype='float32', name='posts')
    x = AddSingleLayer(inputs, 64, (7,7))
    
    #filters = [64,128,256,512,1024]
    filters = [64,128, 256]
    for num_filters in filters:
        #x = AddSingleLayer(x, num_filters)
        x = add_block(x, num_filters=num_filters)
    #x = AddSingleLayer(x, 256)
    x = AddSingleLayer(x, 512)
    x = layers.SpatialDropout2D(0.2)(x)
    x = AddSingleLayer(x, 1024)
    x = layers.SpatialDropout2D(0.3)(x)
    # x = layers.MaxPooling2D(pool_size = 2)(x)
    # x = AddSingleLayer(x, 1024)
    # x = layers.SpatialDropout2D(0.3)(x)
    
    cho = BranchBlock(x,128,len(ja2label),'DenseCho2')
    jung = BranchBlock(x,128,len(mo2label),'DenseJung2')
    jong = BranchBlock(x,128,len(ba2label),'DenseJong2')
    
    model = tf.keras.Model(inputs, [cho, jung, jong])
    return model

model = create_model()
model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])
        
    

In [25]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ posts (InputLayer)  │ (None, 64, 64, 3) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 64, 64,    │      9,472 │ posts[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 64, 64,    │     36,928 │ activation_6[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_3  │ (None, 21, 21,    │        640 │ activation_7[0][… │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 21, 21,    │     73,856 │ depthwise_conv2d… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 21, 21,    │        512 │ conv2d_8[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 21, 21,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_4  │ (None, 7, 7, 128) │      1,280 │ activation_8[0][… │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 7, 7, 256) │    295,168 │ depthwise_conv2d… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 7, 7, 256) │      1,024 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_9        │ (None, 7, 7, 256) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_5  │ (None, 2, 2, 256) │      2,560 │ activation_9[0][… │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 2, 2, 512) │  1,180,160 │ depthwise_conv2d… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2, 2, 512) │      2,048 │ conv2d_10[0][0] 

 Total params: 6,655,632 (25.39 MB)

 Trainable params: 6,651,536 (25.37 MB)

 Non-trainable params: 4,096 (16.00 KB)

In [26]:
model.load_weights(WEIGHT_FILE)

/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:396: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 74 variables. 
  trackable.load_own_variables(weights_store.get(inner_path))


In [27]:
save_dir = SAVE_DIR
checkPoint_path = WEIGHT_FILE
#from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
# 3번 반복내에 validation loss가 줄어들지 않으면 learning rate를 0.2 감소
#lr_cb = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, mode='min', verbose=1)
# 5번 반복내에 validation loss가 줄어들지 않으면 강제종료
#st_cb = EarlyStopping(monitor='val_loss', patience=5, mode='min', verbose=1)

cp_callback = keras.callbacks.ModelCheckpoint(filepath = checkPoint_path, save_weights_only=True, save_best_only=True, monitor = 'loss')

#model.fit(dataset, validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback])
model.fit(dataset, batch_size = 16, epochs = 200, callbacks=[cp_callback], validation_data= validDtaset)
#model.train_on_batch(dataset)

Epoch 1/200
  20003/Unknown 371s 18ms/step - DenseCho2_accuracy: 0.9091 - DenseJong2_accuracy: 0.9366 - DenseJung2_accuracy: 0.8935 - loss: 0.8641

2024-05-28 16:22:01.470520: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-05-28 16:22:01.470904: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]


20004/20004 ━━━━━━━━━━━━━━━━━━━━ 388s 19ms/step - DenseCho2_accuracy: 0.9091 - DenseJong2_accuracy: 0.9366 - DenseJung2_accuracy: 0.8935 - loss: 0.8641 - val_DenseCho2_accuracy: 0.6179 - val_DenseJong2_accuracy: 0.4648 - val_DenseJung2_accuracy: 0.4188 - val_loss: 5.3465
Epoch 2/200


2024-05-28 16:22:18.162216: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-05-28 16:22:18.162280: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-05-28 16:22:18.162309: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 11862786022796523453
2024-05-28 16:22:18.162324: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 15739194419232518083


20003/20004 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - DenseCho2_accuracy: 0.9009 - DenseJong2_accuracy: 0.9346 - DenseJung2_accuracy: 0.8939 - loss: 0.8961

...

20001/20004 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - DenseCho2_accuracy: 0.9043 - DenseJong2_accuracy: 0.9333 - DenseJung2_accuracy: 0.8915 - loss: 0.8995

2024-05-28 17:33:50.564650: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-05-28 17:33:50.564687: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]


20004/20004 ━━━━━━━━━━━━━━━━━━━━ 340s 17ms/step - DenseCho2_accuracy: 0.9043 - DenseJong2_accuracy: 0.9333 - DenseJung2_accuracy: 0.8915 - loss: 0.8995 - val_DenseCho2_accuracy: 0.6081 - val_DenseJong2_accuracy: 0.5737 - val_DenseJung2_accuracy: 0.4423 - val_loss: 5.8371
Epoch 14/200


2024-05-28 17:34:06.368053: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-05-28 17:34:06.368095: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-05-28 17:34:06.368106: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 11862786022796523453
2024-05-28 17:34:06.368111: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 15739194419232518083
2024-05-28 17:34:06.368116: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 3491672860133448646
2024-05-28 17:34:06.368138: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1642376794032108316


19269/20004 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - DenseCho2_accuracy: 0.9067 - DenseJong2_accuracy: 0.9334 - DenseJung2_accuracy: 0.8991 - loss: 0.8599

In [13]:
#model.save("./testModel.h5")
model.load_weights(WEIGHT_FILE)
model.save('./backbone1.keras')
keras.applications.EfficientNetV2B3